# Netflix Churn Predictor — Data Cleaning

**Goal:** Clean all 6 raw datasets so they're ready to merge and model.

### Datasets
| File | What it contains |
|---|---|
| `users.csv` | One row per user — demographics, plan, active status |
| `movies.csv` | One row per movie/show — metadata |
| `watch_history.csv` | One row per viewing session |
| `reviews.csv` | One row per user review |
| `recommendation_logs.csv` | One row per recommendation shown to a user |
| `search_logs.csv` | One row per search a user made |

### Cleaning checklist (applied to every dataset)
1. **First look** — shape, column names, data types, sample rows
2. **Missing values** — how many? which columns?
3. **Duplicates** — any repeated rows?
4. **Fix data types** — dates as dates, booleans as booleans, numbers as numbers
5. **Dataset-specific checks** — invalid values, ranges, etc.
6. **Save** cleaned file to `clean_data/`

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os

# Make a folder for the cleaned files
os.makedirs('cleaned_data', exist_ok=True)

print('Libraries loaded!')

Libraries loaded!


---
## 1. Load All Datasets

Load everything upfront so we can compare across files later.

In [2]:
users        = pd.read_csv('orig_data/users.csv')
movies       = pd.read_csv('orig_data/movies.csv')
watch        = pd.read_csv('orig_data/watch_history.csv')
reviews      = pd.read_csv('orig_data/reviews.csv')
recs         = pd.read_csv('orig_data/recommendation_logs.csv')
searches     = pd.read_csv('orig_data/search_logs.csv')

# Quick size overview
for name, df in [('users', users), ('movies', movies), ('watch', watch),
                 ('reviews', reviews), ('recs', recs), ('searches', searches)]:
    print(f'{name:12s}: {df.shape[0]:>7,} rows  x  {df.shape[1]:>3} columns')

users       :  10,300 rows  x   16 columns
movies      :   1,040 rows  x   18 columns
watch       : 105,000 rows  x   12 columns
reviews     :  15,450 rows  x   12 columns
recs        :  52,000 rows  x   11 columns
searches    :  26,500 rows  x   11 columns


---
## 2. users.csv

contains `is_active`, which is our churn label.

### 2.1 First look

In [3]:
users.head()

,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,subscription_start_date,is_active,monthly_spend,primary_device,household_size,created_at
0,user_00001,figueroajohn@example.org,Erica,Garza,43.0,Male,USA,Massachusetts,North Jefferyhaven,Basic,2024-04-08,True,36.06,Laptop,1.0,2023-04-01 14:40:50.540242
1,user_00002,blakeerik@example.com,Joshua,Bernard,38.0,Male,USA,Texas,North Noahstad,Premium+,2024-05-24,True,14.59,Desktop,2.0,2024-10-10 15:39:11.030515
2,user_00003,smiller@example.net,Barbara,Williams,32.0,Female,USA,Michigan,Traciebury,Standard,2023-09-22,False,11.71,Desktop,3.0,2024-06-29 14:27:49.560875
3,user_00004,mitchellclark@example.com,Chelsea,Ferguson,11.0,Male,USA,Ohio,South Noah,Standard,2024-08-21,True,28.56,Laptop,2.0,2023-04-11 01:01:59.614841
4,user_00005,richard13@example.net,Jason,Foster,21.0,Female,USA,Arizona,West Donald,Standard,2024-10-28,True,9.54,Desktop,6.0,2025-04-12 19:59:30.137806


In [4]:
users.info()  # shows column names, data types, and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  10300 non-null  str    
 1   email                    10300 non-null  str    
 2   first_name               10300 non-null  str    
 3   last_name                10300 non-null  str    
 4   age                      9071 non-null   float64
 5   gender                   9476 non-null   str    
 6   country                  10300 non-null  str    
 7   state_province           10300 non-null  str    
 8   city                     10300 non-null  str    
 9   subscription_plan        10300 non-null  str    
 10  subscription_start_date  10300 non-null  str    
 11  is_active                10300 non-null  bool   
 12  monthly_spend            9283 non-null   float64
 13  primary_device           10300 non-null  str    
 14  household_size           8755 non

age, gender, monthly_spend, and household size have null values.

must check if:
- there's missing values (fill or drop)
- there's incorrect data types 
- is_active is trustworthy bc it will be the churn label of the model

In [5]:
users.describe()  # stats for numeric columns

,age,monthly_spend,household_size
count,9071.000000,9283.000000,8755.000000
mean,35.039466,22.146445,2.863392
std,12.580667,65.723824,1.563251
min,-7.000000,0.110000,1.000000
25%,27.000000,7.745000,2.000000
50%,35.000000,13.530000,2.000000
75%,43.000000,21.620000,4.000000
max,109.000000,997.800000,8.000000


### 2.2 Missing values

In [6]:
missing = users.isnull().sum()
missing_pct = (missing / len(users) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

,missing_count,missing_%
age,1229,11.93
gender,824,8.00
monthly_spend,1017,9.87
household_size,1545,15.00


In [ ]:
# TODO: for each column with missing values, decide:
# - drop the row (if it's a critical column like user_id)
# - fill with a default (e.g. 'Unknown' for text, median for numbers)
# - leave it (if missing means something, like no state for non-USA users)

# Example: fill missing age with median
# users['age'] = users['age'].fillna(users['age'].median())

# Example: drop rows where user_id is missing
# users = users.dropna(subset=['user_id'])

#### Age

In [7]:
users['age'].describe()


count    9071.000000
mean       35.039466
std        12.580667
min        -7.000000
25%        27.000000
50%        35.000000
75%        43.000000
max       109.000000
Name: age, dtype: float64

In [8]:
print('Ages below 13:', users[users['age'] < 13].shape[0])
print('Ages above 110:', users[users['age'] > 110].shape[0])


Ages below 13: 296
Ages above 110: 0


In [9]:
# Replace invalid ages with group mean
plan_age_means = users[users['age'] >= 13].groupby('subscription_plan')['age'].mean()

users.loc[users['age'] < 13, 'age'] = users[users['age'] < 13]['subscription_plan'].map(plan_age_means)

In [10]:
# Verify no more invalid ages
print('Invalid ages remaining:', users[users['age'] < 13].shape[0])

Invalid ages remaining: 0


In [11]:
users.groupby('subscription_plan')['age'].count()


subscription_plan
Basic       1777
Premium     3193
Premium+     907
Standard    3194
Name: age, dtype: int64

In [12]:
users['age'] = users.groupby('subscription_plan')['age'].transform(
    lambda x: x.fillna(x.mean())
)

print(users['age'].isnull().sum())

0


I chose group mean by subscription plan because age likely correlates with plan type, premium users skew for older and basic skew for younger

#### Gender

In [13]:
# Fill null values with 'Unknown'
users['gender'] = users['gender'].fillna('Unknown')

users['gender'].value_counts(dropna=False)

gender
Female               4324
Male                 4228
Unknown               824
Prefer not to say     471
Other                 453
Name: count, dtype: int64

#### Monthly spend

In [14]:
# use IQR to find outliers
Q1 = users['monthly_spend'].quantile(0.25)
Q3 = users['monthly_spend'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

outliers = users[(users['monthly_spend'] < lower_bound) | (users['monthly_spend'] > upper_bound)]
print(f"\nNumber of outliers: {len(outliers)}")
outliers[['user_id', 'monthly_spend']].head(10)


Q1: 7.745, Q3: 21.62, IQR: 13.875
Lower bound: -13.07
Upper bound: 42.43

Number of outliers: 361


,user_id,monthly_spend
10,user_00011,556.13
65,user_00066,50.72
74,user_00075,44.32
117,user_00118,50.48
160,user_00161,284.95
194,user_00195,43.11
252,user_00253,375.30
257,user_00258,859.24
330,user_00331,49.16
341,user_00342,46.97


In [15]:
users[users['monthly_spend'] > 42.43]['subscription_plan'].value_counts()


subscription_plan
Standard    150
Premium     127
Basic        52
Premium+     32
Name: count, dtype: int64

In [16]:
plan_means = users[users['monthly_spend'] <= 42.43].groupby('subscription_plan')['monthly_spend'].mean()
print(plan_means)


subscription_plan
Basic       14.469887
Premium     14.696362
Premium+    14.713912
Standard    15.046033
Name: monthly_spend, dtype: float64


In [17]:
users.loc[users['monthly_spend'].isnull(), 'monthly_spend'] = users[users['monthly_spend'].isnull()]['subscription_plan'].map(plan_means)

In [18]:
users.groupby('subscription_plan')['monthly_spend'].describe()


,count,mean,std,min,25%,50%,75%,max
subscription_plan,,,,,,,,
Basic,2020.0,19.861231,58.037272,0.25,7.975,14.469887,19.3525,954.98
Premium,3619.0,21.030316,62.301130,0.11,8.285,14.696362,20.4750,997.80
Premium+,1036.0,19.784071,50.302096,0.15,8.260,14.713912,20.1875,836.40
Standard,3625.0,23.141315,67.808395,0.11,8.620,15.046033,20.9300,984.45


#### Household size

In [19]:
users['household_size'].describe()

count    8755.000000
mean        2.863392
std         1.563251
min         1.000000
25%         2.000000
50%         2.000000
75%         4.000000
max         8.000000
Name: household_size, dtype: float64

In [20]:
users['household_size'] = users.groupby('subscription_plan')['household_size'].transform(
    lambda x: x.fillna(x.mean())
)

print(users['household_size'].isnull().sum())


0


In [21]:
users.isnull().sum()


user_id                    0
email                      0
first_name                 0
last_name                  0
age                        0
gender                     0
country                    0
state_province             0
city                       0
subscription_plan          0
subscription_start_date    0
is_active                  0
monthly_spend              0
primary_device             0
household_size             0
created_at                 0
dtype: int64

### 2.3 Duplicates

In [22]:
print('Duplicate rows:', users.duplicated().sum())
print('Duplicate user_ids:', users['user_id'].duplicated().sum())

Duplicate rows: 300
Duplicate user_ids: 300


In [23]:
users = users.drop_duplicates(subset='user_id', keep='first')

### 2.4 Fix data types

In [24]:
# Convert date columns from string → datetime
users['subscription_start_date'] = pd.to_datetime(users['subscription_start_date'])
users['created_at']              = pd.to_datetime(users['created_at'])

# is_active is read as string 'True'/'False' — convert to boolean
users['is_active'] = users['is_active'].map({'True': True, 'False': False})

users.dtypes

user_id                               str
email                                 str
first_name                            str
last_name                             str
age                               float64
gender                                str
country                               str
state_province                        str
city                                  str
subscription_plan                     str
subscription_start_date    datetime64[us]
is_active                          object
monthly_spend                     float64
primary_device                        str
household_size                    float64
created_at                 datetime64[us]
dtype: object

### 2.5 Dataset-specific checks

In [25]:
users_orig = pd.read_csv('orig_data/users.csv')
users['is_active'] = users_orig['is_active'] 

print(users['is_active'].value_counts())
print(f"Churn rate: {(~users['is_active']).mean():.1%}")


is_active
True     8519
False    1481
Name: count, dtype: int64
Churn rate: 14.8%


### 2.6 Save cleaned users

In [27]:
users.to_csv('cleaned_data/users_clean.csv', index=False)
print(f'Saved: {len(users):,} rows')

Saved: 10,000 rows


---
## 3. movies.csv

### 3.1 First look

In [ ]:
movies.head()

In [ ]:
movies.info()

### 3.2 Missing values

In [ ]:
missing = movies.isnull().sum()
missing_pct = (missing / len(movies) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

In [ ]:
# genre_secondary, imdb_rating, production_budget, box_office_revenue,
# number_of_seasons, number_of_episodes are likely to have missing values — that's okay
# (e.g. a movie won't have number_of_seasons)

# Fill missing genre_secondary with 'None'
movies['genre_secondary'] = movies['genre_secondary'].fillna('None')

### 3.3 Duplicates

In [ ]:
print('Duplicate rows:', movies.duplicated().sum())
print('Duplicate movie_ids:', movies['movie_id'].duplicated().sum())

### 3.4 Fix data types

In [ ]:
movies['added_to_platform']  = pd.to_datetime(movies['added_to_platform'])
movies['is_netflix_original'] = movies['is_netflix_original'].map({'True': True, 'False': False})
movies['content_warning']     = movies['content_warning'].map({'True': True, 'False': False})

movies.dtypes

### 3.5 Dataset-specific checks

In [ ]:
# IMDB rating should be 0–10
print('IMDB ratings out of range:', movies[(movies['imdb_rating'] < 0) | (movies['imdb_rating'] > 10)].shape[0])

# Duration should be positive
print('Negative durations:', movies[movies['duration_minutes'] <= 0].shape[0])

# Release year sanity check
print('Release year range:', movies['release_year'].min(), '—', movies['release_year'].max())

### 3.6 Save cleaned movies

In [ ]:
movies.to_csv('clean_data/movies_clean.csv', index=False)
print(f'Saved: {len(movies):,} rows')

---
## 4. watch_history.csv

### 4.1 First look

In [ ]:
watch.head()

In [ ]:
watch.info()

### 4.2 Missing values

In [ ]:
missing = watch.isnull().sum()
missing_pct = (missing / len(watch) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

In [ ]:
# user_rating is often blank — user didn't rate the session
# Leave as NaN (it means 'no rating given')

### 4.3 Duplicates

In [ ]:
print('Duplicate session_ids:', watch['session_id'].duplicated().sum())

### 4.4 Fix data types

In [ ]:
watch['watch_date']  = pd.to_datetime(watch['watch_date'])
watch['is_download'] = watch['is_download'].map({'True': True, 'False': False})

watch.dtypes

### 4.5 Dataset-specific checks

In [ ]:
# watch_duration_minutes should be positive
print('Negative/zero durations:', watch[watch['watch_duration_minutes'] <= 0].shape[0])

# progress_percentage should be 0–100
print('Progress out of range:', watch[(watch['progress_percentage'] < 0) | (watch['progress_percentage'] > 100)].shape[0])

# user_rating should be 1–5 if present
print('Invalid user ratings:', watch[(watch['user_rating'].notna()) & (~watch['user_rating'].isin([1,2,3,4,5]))].shape[0])

In [ ]:
# Check all user_ids in watch_history exist in users
orphan_users = ~watch['user_id'].isin(users['user_id'])
print('Watch sessions with unknown user_id:', orphan_users.sum())

### 4.6 Save cleaned watch history

In [ ]:
watch.to_csv('clean_data/watch_history_clean.csv', index=False)
print(f'Saved: {len(watch):,} rows')

---
## 5. reviews.csv

### 5.1 First look

In [ ]:
reviews.head()

In [ ]:
reviews.info()

### 5.2 Missing values

In [ ]:
missing = reviews.isnull().sum()
missing_pct = (missing / len(reviews) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

### 5.3 Duplicates

In [ ]:
print('Duplicate review_ids:', reviews['review_id'].duplicated().sum())
# A user shouldn't review the same movie twice
print('Same user reviewing same movie twice:', reviews.duplicated(subset=['user_id', 'movie_id']).sum())

### 5.4 Fix data types

In [ ]:
reviews['review_date']       = pd.to_datetime(reviews['review_date'])
reviews['is_verified_watch'] = reviews['is_verified_watch'].map({'True': True, 'False': False})

reviews.dtypes

### 5.5 Dataset-specific checks

In [ ]:
# Rating should be 1–5
print('Invalid ratings:', reviews[~reviews['rating'].isin([1,2,3,4,5])].shape[0])

# helpful_votes can't exceed total_votes
bad_votes = reviews[reviews['helpful_votes'] > reviews['total_votes']]
print('helpful_votes > total_votes:', bad_votes.shape[0])

# Sentiment should match known values
print('Sentiment values:', reviews['sentiment'].value_counts())

### 5.6 Save cleaned reviews

In [ ]:
reviews.to_csv('clean_data/reviews_clean.csv', index=False)
print(f'Saved: {len(reviews):,} rows')

---
## 6. recommendation_logs.csv

### 6.1 First look

In [ ]:
recs.head()

In [ ]:
recs.info()

### 6.2 Missing values

In [ ]:
missing = recs.isnull().sum()
missing_pct = (missing / len(recs) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

In [ ]:
# recommendation_score can be missing for some recommendation types — check which ones
recs[recs['recommendation_score'].isnull()]['recommendation_type'].value_counts()

### 6.3 Duplicates

In [ ]:
print('Duplicate recommendation_ids:', recs['recommendation_id'].duplicated().sum())

### 6.4 Fix data types

In [ ]:
recs['recommendation_date'] = pd.to_datetime(recs['recommendation_date'])
recs['was_clicked']         = recs['was_clicked'].map({'True': True, 'False': False})

recs.dtypes

### 6.5 Dataset-specific checks

In [ ]:
# recommendation_score should be 0–1
bad_scores = recs[(recs['recommendation_score'].notna()) & 
                  ((recs['recommendation_score'] < 0) | (recs['recommendation_score'] > 1))]
print('Scores out of 0–1 range:', bad_scores.shape[0])

# Check recommendation types
print('\nRecommendation types:')
print(recs['recommendation_type'].value_counts())

### 6.6 Save cleaned recommendations

In [ ]:
recs.to_csv('clean_data/recommendation_logs_clean.csv', index=False)
print(f'Saved: {len(recs):,} rows')

---
## 7. search_logs.csv

### 7.1 First look

In [ ]:
searches.head()

In [ ]:
searches.info()

### 7.2 Missing values

In [ ]:
missing = searches.isnull().sum()
missing_pct = (missing / len(searches) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

### 7.3 Duplicates

In [ ]:
print('Duplicate search_ids:', searches['search_id'].duplicated().sum())

### 7.4 Fix data types

In [ ]:
searches['search_date']  = pd.to_datetime(searches['search_date'])
searches['had_typo']     = searches['had_typo'].map({'True': True, 'False': False})
searches['used_filters'] = searches['used_filters'].map({'True': True, 'False': False})

searches.dtypes

### 7.5 Dataset-specific checks

In [ ]:
# search_duration_seconds should be positive
print('Negative search duration:', searches[searches['search_duration_seconds'] <= 0].shape[0])

# results_returned should be >= 0
print('Negative results returned:', searches[searches['results_returned'] < 0].shape[0])

# clicked_result_position should be <= results_returned (can't click result 5 if only 3 returned)
bad_clicks = searches[searches['clicked_result_position'] > searches['results_returned']]
print('Clicked position > results returned:', bad_clicks.shape[0])

### 7.6 Save cleaned searches

In [ ]:
searches.to_csv('clean_data/search_logs_clean.csv', index=False)
print(f'Saved: {len(searches):,} rows')

---
## 8. Cross-Dataset Validation

Make sure IDs are consistent across files — every `user_id` and `movie_id` in the log files should exist in `users` and `movies`.

In [ ]:
valid_users  = set(users['user_id'])
valid_movies = set(movies['movie_id'])

checks = [
    ('watch_history',       watch,    'user_id',  valid_users),
    ('watch_history',       watch,    'movie_id', valid_movies),
    ('reviews',             reviews,  'user_id',  valid_users),
    ('reviews',             reviews,  'movie_id', valid_movies),
    ('recommendation_logs', recs,     'user_id',  valid_users),
    ('recommendation_logs', recs,     'movie_id', valid_movies),
    ('search_logs',         searches, 'user_id',  valid_users),
]

for dataset, df, col, valid_ids in checks:
    orphans = (~df[col].isin(valid_ids)).sum()
    status  = 'OK' if orphans == 0 else f'WARNING: {orphans:,} orphan rows'
    print(f'{dataset:25s} | {col:10s} | {status}')

---
## 9. Summary

Run this cell at the end to confirm all cleaned files were saved.

In [ ]:
print('=== Cleaned files ===')
for f in sorted(os.listdir('clean_data')):
    df = pd.read_csv(f'clean_data/{f}')
    print(f'  {f:40s}  {len(df):>8,} rows')